In [ ]:
# Cell 1
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.data_collection.lob_reconstructor import *
from src.features.ofi_features import compute_all_features
from src.backtest.signal_backtest import OFIBacktester, parameter_sweep

plt.style.use('dark_background')
%matplotlib inline

RUN_ID = "YOUR_RUN_ID_HERE"
DATA_DIR = Path("../data")
depth  = load_depth_data(RUN_ID, DATA_DIR)
trades = load_trade_data(RUN_ID, DATA_DIR)
depth  = add_derived_columns(depth)
merged = merge_depth_and_trades(depth, trades)
df     = compute_all_features(merged)

In [ ]:
# Cell 2 — Run backtest
bt     = OFIBacktester(entry_threshold=1.5, holding_period=10, taker_fee_bps=4.0)
trades_result = bt.run(df)

In [ ]:
# Cell 3 — Equity curve
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(range(len(trades_result)), trades_result['cumulative_net_pnl'],
             color='#00ff88', lw=1.5, label='Net PnL (after TC)')
axes[0].plot(range(len(trades_result)), trades_result['cumulative_gross_pnl'],
             color='#4488ff', lw=1, ls='--', alpha=0.6, label='Gross PnL (before TC)')
axes[0].fill_between(range(len(trades_result)), 0,
                     trades_result['cumulative_net_pnl'], alpha=0.15, color='#00ff88')
axes[0].axhline(0, color='white', lw=0.5)
axes[0].set_title(
    f"OFI Signal Equity Curve  |  "
    f"Trades: {len(trades_result)}  |  "
    f"Win rate: {trades_result['profitable'].mean():.1%}  |  "
    f"Net PnL: {trades_result['net_pnl'].sum():.5f}"
)
axes[0].legend(); axes[0].set_ylabel('Cumulative PnL')

pnl_colors = ['#00ff88' if x > 0 else '#ff4444' for x in trades_result['net_pnl']]
axes[1].bar(range(len(trades_result)), trades_result['net_pnl'],
            color=pnl_colors, alpha=0.7, width=1.0)
axes[1].axhline(0, color='white', lw=0.5)
axes[1].set_title('Per-Trade Net PnL')
axes[1].set_ylabel('Net PnL per trade')

plt.tight_layout()
plt.savefig('../data/processed/05_equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4 — Parameter sweep heatmap
sweep = parameter_sweep(df)
pivot = sweep.pivot(index='threshold', columns='holding_period', values='sharpe')

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label': 'Sharpe Ratio'})
plt.title('OFI Signal Sharpe by Entry Threshold × Holding Period')
plt.xlabel('Holding period (snapshots)'); plt.ylabel('OFI z-score threshold')
plt.tight_layout()
plt.savefig('../data/processed/05_param_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()